# House Prices: TabNet Approach
**Pipeline**: 前処理 → TabNet (Attention-based DL for Tabular Data) → マルチシード提出

- 決定木ベースのスタッキング (`pipeline.ipynb`) とは別アプローチ
- TabNet: Google Research発のテーブルデータ向けDL。特徴量選択をAttentionで自動学習
- 前処理は共通 (HousePricesPreprocessor) を流用、モデル部分のみ差し替え

## Setup

In [ ]:
import sys, shutil, os, json, zipfile
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ or Path('/content').is_dir()
HAS_GPU = shutil.which('nvidia-smi') is not None
print(f"Environment: {'Colab' if IS_COLAB else 'Local'}, GPU: {HAS_GPU}")

if HAS_GPU:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    for line in result.stdout.split('\n')[:4]:
        print(line)

if IS_COLAB:
    !pip install -q pytorch-tabnet lightgbm
else:
    import importlib
    if importlib.util.find_spec('pytorch_tabnet') is None:
        !pip install pytorch-tabnet

In [ ]:
if IS_COLAB:
    import os
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error
from pytorch_tabnet.tab_model import TabNetRegressor
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch device: {DEVICE}")
print(f"TabNet version: {__import__('pytorch_tabnet').__version__}")

In [ ]:
import os, json, zipfile
from pathlib import Path

def _setup_kaggle_credentials():
    """Ensure kaggle credentials are available."""
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        return
    # Try Colab Secrets
    try:
        from google.colab import userdata
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        if username and key:
            kaggle_dir.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_text(json.dumps({'username': username, 'key': key}))
            kaggle_json.chmod(0o600)
            print(f"Created kaggle.json from Colab Secrets")
            return
    except Exception:
        pass
    # Try Google Drive
    drive_kaggle = Path('/content/drive/MyDrive/.kaggle/kaggle.json')
    if drive_kaggle.exists():
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy2(drive_kaggle, kaggle_json)
        kaggle_json.chmod(0o600)
        print(f"Copied kaggle.json from Google Drive")
        return
    raise FileNotFoundError(
        "kaggle.json not found. Options:\n"
        "  1. Colab Secrets: Add KAGGLE_USERNAME and KAGGLE_KEY\n"
        "  2. Google Drive: Place kaggle.json at Drive/MyDrive/.kaggle/kaggle.json")

if IS_COLAB:
    DATA_DIR = Path("data")
else:
    candidates = [
        Path("data"),
        Path("house-prices/data"),
    ]
    try:
        candidates.insert(0, Path(__vsc_ipynb_file__).parent / "data")
    except NameError:
        pass
    DATA_DIR = Path("data")
    for candidate in candidates:
        if (candidate / "train.csv").exists():
            DATA_DIR = candidate
            break

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

if not TRAIN_CSV.exists():
    print("Data not found. Downloading via Kaggle CLI...")
    _setup_kaggle_credentials()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c house-prices-advanced-regression-techniques -p {DATA_DIR}
    zip_path = DATA_DIR / 'house-prices-advanced-regression-techniques.zip'
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    zip_path.unlink()

print(f"DATA_DIR: {DATA_DIR}")
print(f"Train: {TRAIN_CSV.exists()}, Test: {TEST_CSV.exists()}")

# Output directories
if IS_COLAB:
    SUBMISSION_DIR = '/content/submissions'
    FIGURES_DIR = '/content/figures'
else:
    _project_root = Path(DATA_DIR).resolve().parent if Path(DATA_DIR).name == 'data' else Path('.')
    SUBMISSION_DIR = str(_project_root / 'submissions')
    FIGURES_DIR = str(_project_root / 'figures')

os.makedirs(SUBMISSION_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

## Data Loading

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

# Outlier removal (GrLivArea > 4000 & SalePrice < 300000)
outlier_mask = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
print(f"Outliers removed: {outlier_mask.sum()}")
train = train[~outlier_mask].reset_index(drop=True)

y = np.log1p(train['SalePrice'])
X = train.drop(columns=['SalePrice'])
X_test = test.copy()
test_ids = test['Id']

print(f"Train: {X.shape}, Test: {X_test.shape}")

## Constants

In [ ]:
CLIP_MIN = 35000
CLIP_MAX = 600000

NEIGHBORHOOD_COORDS = {
    "Blmngtn": (42.062736, -93.641598), "Blueste": (42.055300, -93.648000),
    "BrDale": (42.052702, -93.627195), "BrkSide": (42.033590, -93.627692),
    "ClearCr": (42.060438, -93.621175), "CollgCr": (42.021678, -93.685793),
    "Crawfor": (42.015962, -93.640486), "Edwards": (42.022110, -93.663703),
    "Gilbert": (42.063379, -93.647255), "Greens": (42.063150, -93.642700),
    "GrnHill": (42.061000, -93.643500), "IDOTRR": (42.022632, -93.619070),
    "Landmrk": (42.030000, -93.620000), "MeadowV": (42.032428, -93.641709),
    "Mitchel": (42.033900, -93.614700), "NAmes": (42.044800, -93.650500),
    "NoRidge": (42.050760, -93.656810), "NPkVill": (42.050000, -93.625000),
    "NridgHt": (42.060200, -93.656200), "NWAmes": (42.051000, -93.659100),
    "OldTown": (42.028270, -93.613600), "SWISU": (42.017200, -93.651600),
    "Sawyer": (42.035050, -93.666120), "SawyerW": (42.035300, -93.685700),
    "Somerst": (42.052370, -93.645830), "StoneBr": (42.060000, -93.648500),
    "Timber": (42.017000, -93.681000), "Veenker": (42.040800, -93.651700),
}
AMES_CENTER = (42.05, -93.65)
HIGH_PRICE_CENTER = (42.057, -93.646)

REDUNDANT_COLS = [
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea',
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath',
    'YearBuilt', 'GarageArea'
]

## Preprocessor (pipeline.ipynb 共通)

In [ ]:
class HousePricesPreprocessor(BaseEstimator, TransformerMixin):
    """Shared preprocessor from pipeline.ipynb"""

    NA_MEANS_NONE = [
        'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
        'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
        'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
        'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
    ]

    ORDINAL_MAP = {
        'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
        'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'FireplaceQu': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
        'PoolQC': {'None': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
        'Fence': {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
        'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
        'LandSlope': {'Sev': 1, 'Mod': 2, 'Gtl': 3},
        'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
        'CentralAir': {'N': 0, 'Y': 1},
        'Street': {'Grvl': 0, 'Pave': 1},
        'Utilities': {'ELO': 0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3},
    }

    def __init__(self, selected_features=None):
        self.selected_features = selected_features

    def _fill_missing(self, df):
        for col in self.NA_MEANS_NONE:
            if col in df.columns:
                df[col] = df[col].fillna('None')
        num_cols = df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if df[col].isnull().any():
                if col in ['LotFrontage']:
                    df[col] = df[col].fillna(df[col].median())
                else:
                    df[col] = df[col].fillna(0)
        obj_cols = df.select_dtypes(include=['object']).columns
        for col in obj_cols:
            if df[col].isnull().any():
                df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'None')
        return df

    def _ordinal_encode(self, df):
        for col, mapping in self.ORDINAL_MAP.items():
            if col in df.columns:
                df[col] = df[col].map(mapping).fillna(0).astype(int)
        return df

    def _add_geo_features(self, df):
        if 'Neighborhood' in df.columns:
            df['Latitude'] = df['Neighborhood'].map({k: v[0] for k, v in NEIGHBORHOOD_COORDS.items()})
            df['Longitude'] = df['Neighborhood'].map({k: v[1] for k, v in NEIGHBORHOOD_COORDS.items()})
            df['Latitude'] = df['Latitude'].fillna(AMES_CENTER[0])
            df['Longitude'] = df['Longitude'].fillna(AMES_CENTER[1])
            df['Dist_to_Center'] = np.sqrt(
                (df['Latitude'] - AMES_CENTER[0])**2 + (df['Longitude'] - AMES_CENTER[1])**2)
            df['Dist_to_HighPrice_Center'] = np.sqrt(
                (df['Latitude'] - HIGH_PRICE_CENTER[0])**2 + (df['Longitude'] - HIGH_PRICE_CENTER[1])**2)
        return df

    def _add_domain_features(self, df):
        df['TotalSF'] = df.get('TotalBsmtSF', 0) + df.get('1stFlrSF', 0) + df.get('2ndFlrSF', 0)
        if 'TotalBsmtSF' in df.columns:
            df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = (df.get('FullBath', 0) + df.get('HalfBath', 0)
                          + df.get('BsmtFullBath', 0) + df.get('BsmtHalfBath', 0))
        if 'FullBath' in df.columns:
            df['TotalBath'] = df['FullBath'] + df['HalfBath'] + df['BsmtFullBath'] + df['BsmtHalfBath']
        df['HouseAge'] = df['YrSold'] - df.get('YearBuilt', df['YrSold'])
        if 'YearBuilt' in df.columns:
            df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df['RemodAge'] = df['YrSold'] - df.get('YearRemodAdd', df['YrSold'])
        if 'YearRemodAdd' in df.columns:
            df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']

        porch_cols = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
        df['TotalPorchSF'] = sum(df.get(c, 0) for c in porch_cols)

        total_sf = df.get('TotalSF', 1)
        gr_liv = df.get('GrLivArea', df.get('TotalSF', 1))
        df['Living_Space_Ratio'] = np.where(total_sf > 0, gr_liv / total_sf, 1.0)

        bsmt_sf = df.get('TotalBsmtSF', 0)
        second_sf = df.get('2ndFlrSF', 0)
        df['Luxury_Space_Index'] = np.where(
            total_sf > 0,
            (bsmt_sf * 0.3 + second_sf * 0.4) / total_sf, 0.0)

        df['IsNew'] = (df['HouseAge'] == 0).astype(int)
        df['HasRemod'] = (df['RemodAge'] >= 0).astype(int)
        df['HasGarage'] = (df.get('GarageCars', 0) > 0).astype(int)
        df['HasBsmt'] = (df.get('TotalBsmtSF', 0) > 0).astype(int)
        df['Has2ndFlr'] = (df.get('2ndFlrSF', 0) > 0).astype(int)
        df['HasPool'] = (df.get('PoolArea', 0) > 0).astype(int)

        # QOL Score
        qual_cols = ['OverallQual', 'ExterQual', 'KitchenQual']
        cond_cols = ['OverallCond', 'ExterCond']
        q_sum = sum(df.get(c, 0) for c in qual_cols)
        c_sum = sum(df.get(c, 0) for c in cond_cols)
        func_val = df.get('Functional', 8)
        df['QOL_Score'] = q_sum * 0.5 + c_sum * 0.3 + func_val * 0.2

        return df

    def _drop_redundant(self, df):
        drop_cols = [c for c in REDUNDANT_COLS if c in df.columns]
        df = df.drop(columns=drop_cols)
        df = df.drop(columns=['Id'], errors='ignore')
        return df

    def _fix_skewness(self, df):
        num_cols = df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if df[col].nunique() > 2 and df[col].skew() > 0.7:
                df[col] = np.log1p(df[col].clip(lower=0))
        return df

    def fit_transform(self, X, y=None):
        df = X.copy()
        df = self._fill_missing(df)
        df = self._ordinal_encode(df)
        df = self._add_geo_features(df)
        df = self._add_domain_features(df)
        df = self._drop_redundant(df)
        df = self._fix_skewness(df)

        # Label encode remaining categoricals
        self.label_encoders_ = {}
        self.cat_columns_ = []
        for col in df.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.label_encoders_[col] = le
            self.cat_columns_.append(col)

        if self.selected_features is not None:
            keep = [c for c in self.selected_features if c in df.columns]
            df = df[keep]

        self.feature_names_ = list(df.columns)
        self.cat_feature_indices_ = [i for i, c in enumerate(self.feature_names_) if c in self.cat_columns_]
        return df

    def transform(self, X):
        df = X.copy()
        df = self._fill_missing(df)
        df = self._ordinal_encode(df)
        df = self._add_geo_features(df)
        df = self._add_domain_features(df)
        df = self._drop_redundant(df)
        df = self._fix_skewness(df)

        for col, le in self.label_encoders_.items():
            if col in df.columns:
                # Handle unseen labels
                known = set(le.classes_)
                df[col] = df[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
                df[col] = le.transform(df[col])

        if self.selected_features is not None:
            keep = [c for c in self.selected_features if c in df.columns]
            df = df[keep]

        return df

## KNN Geo Price Feature

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

def compute_knn_geo_price(X, y, X_test, n_neighbors=10, n_splits=5, seed=42):
    """Leak-free KNN geographic price feature."""
    def _get_coords(df):
        lat = df['Neighborhood'].map({k: v[0] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(AMES_CENTER[0])
        lon = df['Neighborhood'].map({k: v[1] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(AMES_CENTER[1])
        return np.column_stack([lat.values, lon.values])

    coords_train = _get_coords(X)
    coords_test = _get_coords(X_test)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for train_idx, val_idx in kf.split(X):
        knn = KNeighborsRegressor(n_neighbors=n_neighbors)
        knn.fit(coords_train[train_idx], y.values[train_idx])
        oof[val_idx] = knn.predict(coords_train[val_idx])
        test_preds += knn.predict(coords_test) / n_splits

    return oof, test_preds

knn_train, knn_test = compute_knn_geo_price(X, y, X_test)
X['KNN_Geo_Price'] = knn_train
X_test['KNN_Geo_Price'] = knn_test
print(f"KNN Geo Price: train mean={knn_train.mean():.4f}, test mean={knn_test.mean():.4f}")

## Feature Selection (LightGBM importance)

In [ ]:
import lightgbm as lgb

SHAP_TOP_N = 40

# Preprocess all features for importance ranking
_prep_fs = HousePricesPreprocessor(selected_features=None)
_X_fs = _prep_fs.fit_transform(X, y)

_lgb_fs = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.01, max_depth=3,
    num_leaves=7, min_child_samples=50, subsample=0.8,
    colsample_bytree=0.8, random_state=42, verbose=-1
)
_lgb_fs.fit(_X_fs.values.astype(np.float64), y)

_importances = pd.Series(_lgb_fs.feature_importances_, index=_prep_fs.feature_names_)
_importances = _importances.sort_values(ascending=False)
_importances = _importances.drop(labels=[c for c in _importances.index if c.endswith('_te')], errors='ignore')

SELECTED_FEATURES = _importances.head(SHAP_TOP_N).index.tolist()
print(f"Selected {len(SELECTED_FEATURES)} features")
print(SELECTED_FEATURES[:10], '...')

## Preprocess for TabNet

In [ ]:
preprocessor = HousePricesPreprocessor(selected_features=SELECTED_FEATURES)
X_processed = preprocessor.fit_transform(X, y)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.feature_names_
cat_idxs = preprocessor.cat_feature_indices_
cat_dims = [int(X_processed.iloc[:, i].nunique()) for i in cat_idxs]

print(f"Features: {len(feature_names)}, Categorical: {len(cat_idxs)}")
print(f"Cat features: {[feature_names[i] for i in cat_idxs]}")
print(f"Cat dims: {cat_dims}")

# StandardScaler for numeric features (TabNet benefits from scaling)
num_idxs = [i for i in range(len(feature_names)) if i not in cat_idxs]
scaler = StandardScaler()

X_arr = X_processed.values.astype(np.float64)
X_test_arr = X_test_processed.values.astype(np.float64)

X_arr[:, num_idxs] = scaler.fit_transform(X_arr[:, num_idxs])
X_test_arr[:, num_idxs] = scaler.transform(X_test_arr[:, num_idxs])

y_arr = y.values.reshape(-1, 1).astype(np.float64)

print(f"X_arr: {X_arr.shape}, X_test_arr: {X_test_arr.shape}")

## TabNet Configuration

In [ ]:
def get_tabnet_params(seed=42):
    """TabNet hyperparameters tuned for small dataset (1454 rows)."""
    return dict(
        n_d=16,                # Width of decision prediction layer
        n_a=16,                # Width of attention layer
        n_steps=4,             # Number of sequential attention steps
        gamma=1.5,             # Coefficient for feature reusage (higher = more reuse)
        lambda_sparse=1e-3,    # Sparsity regularization
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=1,         # Embedding dim for categoricals
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
        scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts,
        scheduler_params=dict(T_0=50, T_mult=2, eta_min=1e-5),
        mask_type='entmax',    # 'sparsemax' or 'entmax' for attention
        seed=seed,
        verbose=0,
        device_name=DEVICE,
    )

FIT_PARAMS = dict(
    max_epochs=300,
    patience=30,            # Early stopping patience
    batch_size=128,
    virtual_batch_size=64,  # Ghost batch normalization
    num_workers=0,
    drop_last=False,
)

print("TabNet params ready")
print(f"  n_d/n_a=16, n_steps=4, mask=entmax")
print(f"  max_epochs={FIT_PARAMS['max_epochs']}, patience={FIT_PARAMS['patience']}")

## CV Evaluation

In [ ]:
def tabnet_cv(X_arr, y_arr, n_splits=5, seed=42):
    """TabNet cross-validation with early stopping."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X_arr))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_arr)):
        X_train, X_val = X_arr[train_idx], X_arr[val_idx]
        y_train, y_val = y_arr[train_idx], y_arr[val_idx]

        model = TabNetRegressor(**get_tabnet_params(seed=seed + fold))
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric=['rmse'],
            **FIT_PARAMS,
        )

        preds = model.predict(X_val).flatten()
        oof_preds[val_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_arr[val_idx].flatten(), preds))
        fold_scores.append(rmse)
        print(f"  Fold {fold+1}: RMSLE={rmse:.5f} (best epoch: {model.best_epoch})")

    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    overall = np.sqrt(mean_squared_error(y_arr.flatten(), oof_preds))
    print(f"\n  CV Mean: {mean_score:.5f} +/- {std_score:.5f}")
    print(f"  OOF RMSLE: {overall:.5f}")
    return oof_preds, fold_scores

print("Running TabNet CV...")
oof_preds, fold_scores = tabnet_cv(X_arr, y_arr, n_splits=5, seed=42)

## Visualization Suite: TabNet の思考回路を読み解く

3つの可視化で TabNet の推論プロセスを人間に翻訳する:

1. **Attention Mask ヒートマップ** — 各 Step で「何を見ているか」の段階的推論
2. **GBDT vs TabNet 重要度比較** — 決定木と NN の「視点の違い」
3. **Learning Curve** — 収束状況と過学習リスクの直感的把握

In [ ]:
# ============================================================
# Train a diagnostic model (90/10 split for early stopping)
# This model is used for all 3 visualizations below
# ============================================================
from sklearn.model_selection import train_test_split
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib import cm

_X_tr, _X_es, _y_tr, _y_es = train_test_split(
    X_arr, y_arr, test_size=0.1, random_state=42)

model_viz = TabNetRegressor(**get_tabnet_params(seed=42))
model_viz.fit(
    _X_tr, _y_tr,
    eval_set=[(_X_es, _y_es)],
    eval_metric=['rmse'],
    **FIT_PARAMS,
)

print(f"Diagnostic model trained: best_epoch={model_viz.best_epoch}")
print(f"  Train RMSE: {model_viz.history['loss'][-1]:.5f}")
print(f"  Valid RMSE: {model_viz.history['val_0_rmse'][-1]:.5f}")

### 1. Attention Mask ヒートマップ — 段階的推論の可視化
TabNet は n_steps=4 の各ステップで異なる特徴量に「注目」する。
- Step 1 で構造(面積)を見て、Step 2 で品質を見て… というモデルの思考過程を可視化。
- 横軸: 特徴量、縦軸: サンプル。色が濃い = そのステップでその特徴量を強く注視。

In [ ]:
# ============================================================
# VIZ 1: Attention Mask Heatmap (per-step)
# ============================================================
# explain() returns (M_explain, masks_list)
#   M_explain: aggregated mask (n_samples, n_features)
#   masks_list: list of n_steps masks, each (n_samples, n_features)

M_explain, masks = model_viz.explain(X_arr)
n_steps = len(masks)
n_features = len(feature_names)

# --- Upper panel: per-step heatmap (sample x feature) ---
# Sort samples by predicted price for interpretable ordering
preds_viz = model_viz.predict(X_arr).flatten()
sort_idx = np.argsort(preds_viz)

# Show 100 evenly-spaced samples (cheap → expensive)
sample_idx = sort_idx[np.linspace(0, len(sort_idx)-1, 100, dtype=int)]

fig = plt.figure(figsize=(22, 14))
gs = gridspec.GridSpec(2, n_steps + 1, height_ratios=[3, 1.2],
                       width_ratios=[1]*n_steps + [0.05],
                       hspace=0.35, wspace=0.08)

# Per-step heatmaps
for step in range(n_steps):
    ax = fig.add_subplot(gs[0, step])
    mask_data = masks[step][sample_idx]  # (100, n_features)
    im = ax.imshow(mask_data, aspect='auto', cmap='YlOrRd',
                   interpolation='nearest')
    ax.set_title(f'Step {step+1}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Samples (cheap → expensive)' if step == 0 else '')
    ax.set_xlabel('')
    ax.set_xticks([])
    if step > 0:
        ax.set_yticks([])

# Colorbar
cbar_ax = fig.add_subplot(gs[0, -1])
plt.colorbar(im, cax=cbar_ax, label='Attention weight')

# --- Lower panel: per-step average attention (bar chart) ---
# Identify top-10 features by aggregate importance for labeling
agg_importance = M_explain.mean(axis=0)
top_k = 15
top_feat_idx = np.argsort(agg_importance)[-top_k:][::-1]

for step in range(n_steps):
    ax = fig.add_subplot(gs[1, step])
    step_avg = masks[step].mean(axis=0)  # (n_features,)
    # Show only top-15 features
    vals = step_avg[top_feat_idx]
    labels = [feature_names[i] for i in top_feat_idx]
    colors = plt.cm.YlOrRd(vals / (vals.max() + 1e-9))
    ax.barh(range(top_k), vals, color=colors)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(labels if step == 0 else [], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Avg attention', fontsize=9)
    ax.set_title(f'Step {step+1} Top Features', fontsize=10)

fig.suptitle('TabNet Attention Masks: 段階的推論プロセス\n'
             '(上: サンプル別ヒートマップ / 下: Step別 平均注視度)',
             fontsize=15, fontweight='bold', y=0.98)
plt.savefig(f'{FIGURES_DIR}/tabnet_masks.png', dpi=150, bbox_inches='tight')
plt.show()

# Print step-by-step interpretation
print("\n=== Step-by-Step 推論の解釈 ===")
for step in range(n_steps):
    step_avg = masks[step].mean(axis=0)
    top3_idx = np.argsort(step_avg)[-3:][::-1]
    top3 = [(feature_names[i], step_avg[i]) for i in top3_idx]
    print(f"  Step {step+1}: {top3[0][0]} ({top3[0][1]:.3f}), "
          f"{top3[1][0]} ({top3[1][1]:.3f}), {top3[2][0]} ({top3[2][1]:.3f})")

### 2. GBDT vs TabNet 重要度比較 — 決定木と NN の「視点の違い」
同じ特徴量セットに対して、LightGBM (split-based) と TabNet (attention-based) の重視度を比較。

- **GBDT のみ高い**: 離散的な分岐で効く変数 (閾値効果)
- **TabNet のみ高い**: 連続的・非線形な関係を NN が捉えている変数
- **両方高い**: どのモデルでも本質的に重要な変数

In [ ]:
# ============================================================
# VIZ 2: GBDT vs TabNet Feature Importance Comparison
# ============================================================
# LightGBM importance: _importances (computed earlier in feature selection)
# TabNet importance: model_viz.feature_importances_

# Normalize both to [0, 1] for fair comparison
tabnet_imp = pd.Series(model_viz.feature_importances_, index=feature_names)
tabnet_imp_norm = tabnet_imp / (tabnet_imp.max() + 1e-9)

# Get LightGBM importance for the same selected features
lgb_imp = _importances.reindex(feature_names).fillna(0)
lgb_imp_norm = lgb_imp / (lgb_imp.max() + 1e-9)

# Combine into DataFrame
comp_df = pd.DataFrame({
    'LightGBM': lgb_imp_norm,
    'TabNet': tabnet_imp_norm,
}).sort_values('LightGBM', ascending=False)

# Compute difference: positive = TabNet favors more, negative = GBDT favors more
comp_df['diff'] = comp_df['TabNet'] - comp_df['LightGBM']

# --- Plot: butterfly chart (top 25 features) ---
top_n_show = 25
df_plot = comp_df.head(top_n_show).copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 10), gridspec_kw={'width_ratios': [3, 1.2]})

# Left panel: side-by-side bars
ax = axes[0]
y_pos = np.arange(len(df_plot))
bar_height = 0.35

bars_lgb = ax.barh(y_pos + bar_height/2, df_plot['LightGBM'], bar_height,
                    label='LightGBM (GBDT)', color='#2196F3', alpha=0.85)
bars_tab = ax.barh(y_pos - bar_height/2, df_plot['TabNet'], bar_height,
                    label='TabNet (NN)', color='#FF5722', alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Normalized Importance', fontsize=11)
ax.set_title('Feature Importance: LightGBM vs TabNet (Top 25)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.axvline(x=0, color='gray', linewidth=0.5)

# Right panel: difference (TabNet - GBDT)
ax2 = axes[1]
colors = ['#FF5722' if d > 0 else '#2196F3' for d in df_plot['diff']]
ax2.barh(y_pos, df_plot['diff'], color=colors, alpha=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([])
ax2.invert_yaxis()
ax2.set_xlabel('TabNet - GBDT', fontsize=11)
ax2.set_title('Difference', fontsize=13, fontweight='bold')
ax2.axvline(x=0, color='gray', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Insights: features with biggest disagreement
print("\n=== GBDT vs TabNet: 視点の違い ===")
biggest_tabnet = comp_df.nlargest(5, 'diff')
biggest_gbdt = comp_df.nsmallest(5, 'diff')
print("\nTabNet が GBDT より重視 (NN が捉える非線形性):")
for feat, row in biggest_tabnet.iterrows():
    print(f"  {feat:30s}  GBDT={row['LightGBM']:.3f}  TabNet={row['TabNet']:.3f}  diff=+{row['diff']:.3f}")
print("\nGBDT が TabNet より重視 (閾値分岐で効く変数):")
for feat, row in biggest_gbdt.iterrows():
    print(f"  {feat:30s}  GBDT={row['LightGBM']:.3f}  TabNet={row['TabNet']:.3f}  diff={row['diff']:.3f}")

### 3. Learning Curve — 収束状況と過学習リスクの可視化
Train/Valid の RMSE 推移をプロットし、Best Epoch の位置を明示。

- **Train-Valid の乖離**: 過学習の度合いを示す。乖離が大きい = 正則化強化 or データ追加が必要
- **Best Epoch の位置**: Early stopping が適切に機能しているか
- **収束速度**: 学習率やスケジューラの妥当性

In [ ]:
# ============================================================
# VIZ 3: Learning Curve with Early Stopping
# ============================================================
train_loss = model_viz.history['loss']
valid_rmse = model_viz.history['val_0_rmse']
epochs = range(1, len(train_loss) + 1)
best_epoch = model_viz.best_epoch

fig, ax = plt.subplots(figsize=(14, 6))

# Plot curves
ax.plot(epochs, train_loss, label='Train RMSE', color='#2196F3',
        linewidth=2, alpha=0.9)
ax.plot(epochs, valid_rmse, label='Valid RMSE', color='#FF5722',
        linewidth=2, alpha=0.9)

# Best epoch marker
best_val = valid_rmse[best_epoch - 1] if best_epoch <= len(valid_rmse) else valid_rmse[-1]
ax.axvline(x=best_epoch, color='#4CAF50', linewidth=2, linestyle='--',
           label=f'Best Epoch = {best_epoch}', alpha=0.8)
ax.scatter([best_epoch], [best_val], color='#4CAF50', s=120, zorder=5,
           edgecolors='white', linewidth=2)
ax.annotate(f'Best: {best_val:.5f}',
            xy=(best_epoch, best_val),
            xytext=(best_epoch + len(epochs)*0.05, best_val + 0.005),
            fontsize=11, fontweight='bold', color='#4CAF50',
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=1.5))

# Overfitting gap annotation
final_train = train_loss[-1]
final_valid = valid_rmse[-1]
gap = final_valid - final_train
ax.fill_between(epochs, train_loss, valid_rmse,
                alpha=0.1, color='red', label=f'Overfit gap (final: {gap:.4f})')

# Formatting
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('RMSE (log SalePrice)', fontsize=12)
ax.set_title(f'TabNet Learning Curve
'
             f'Best Valid RMSE = {best_val:.5f} @ epoch {best_epoch} / {len(train_loss)}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(1, len(epochs))

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/tabnet_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# Diagnostics
print(f"
=== Learning Curve 診断 ===" )
print(f"  Total epochs:     {len(train_loss)}")
print(f"  Best epoch:       {best_epoch}")
print(f"  Best valid RMSE:  {best_val:.5f}")
print(f"  Final train RMSE: {final_train:.5f}")
print(f"  Final valid RMSE: {final_valid:.5f}")
print(f"  Overfit gap:      {gap:.5f}")
if gap > 0.02:
    print("  --> 過学習の兆候あり。正則化強化 (lambda_sparse↑, dropout↑) or n_d/n_a↓ を検討")
elif gap < 0.005:
    print("  --> 過少学習の可能性。モデル容量拡大 (n_d/n_a↑, n_steps↑) を検討")
else:
    print("  --> 良好なバランス。現行設定を維持")

## Multi-Seed Prediction & Submission

In [ ]:
def tabnet_predict(X_arr, y_arr, X_test_arr, n_splits=5, seed=42):
    """OOF prediction + test prediction with fold averaging."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X_arr))
    test_preds = np.zeros(len(X_test_arr))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_arr)):
        X_train, X_val = X_arr[train_idx], X_arr[val_idx]
        y_train, y_val = y_arr[train_idx], y_arr[val_idx]

        model = TabNetRegressor(**get_tabnet_params(seed=seed + fold))
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric=['rmse'],
            **FIT_PARAMS,
        )

        oof_preds[val_idx] = model.predict(X_val).flatten()
        test_preds += model.predict(X_test_arr).flatten() / n_splits

    cv_score = np.sqrt(mean_squared_error(y_arr.flatten(), oof_preds))
    return oof_preds, test_preds, cv_score

SEEDS = [42, 123, 456]
all_test_preds = []
all_cv_scores = []

for seed in SEEDS:
    print(f"\n=== Seed {seed} ===")
    oof, test_pred, cv = tabnet_predict(X_arr, y_arr, X_test_arr, seed=seed)
    all_test_preds.append(test_pred)
    all_cv_scores.append(cv)
    print(f"  OOF RMSLE: {cv:.5f}")

print(f"\nMean CV: {np.mean(all_cv_scores):.5f} +/- {np.std(all_cv_scores):.5f}")

# Average across seeds
avg_test_preds = np.mean(all_test_preds, axis=0)
final_preds = np.expm1(avg_test_preds)
final_preds = np.clip(final_preds, CLIP_MIN, CLIP_MAX)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': final_preds})
submission.to_csv(f'{SUBMISSION_DIR}/submission_tabnet.csv', index=False)
print(f"\nSubmission saved: {SUBMISSION_DIR}/submission_tabnet.csv")
print(submission.describe())

## Colab → Google Drive Sync

In [ ]:
if IS_COLAB:
    import shutil, glob
    DRIVE_OUTPUT = '/content/drive/MyDrive/kaggle-output/house-prices'
    os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
    os.makedirs(f'{DRIVE_OUTPUT}/figures', exist_ok=True)

    for f in glob.glob(f'{SUBMISSION_DIR}/submission_tabnet*.csv'):
        shutil.copy2(f, f'{DRIVE_OUTPUT}/submissions/')
    for f in glob.glob(f'{FIGURES_DIR}/tabnet_*.png'):
        shutil.copy2(f, f'{DRIVE_OUTPUT}/figures/')

    from google.colab import files
    files.download(f'{SUBMISSION_DIR}/submission_tabnet.csv')
    print("Files synced to Drive and downloaded.")
else:
    print("Local mode: files saved to", SUBMISSION_DIR)

## Kaggle Submit

In [ ]:
import requests

COMPETITION = 'house-prices-advanced-regression-techniques'
SUBMIT_FILE = f'{SUBMISSION_DIR}/submission_tabnet.csv'
SUBMIT_DESC = f'TabNet n_d=16 n_steps=4 entmax multiseed CV={np.mean(all_cv_scores):.5f}'

_setup_kaggle_credentials()
_kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
_kaggle_token = json.loads(_kaggle_json.read_text()).get('key', '')
_headers = {'Authorization': f'Bearer {_kaggle_token}', 'Content-Type': 'application/json'}
_file_size = os.path.getsize(SUBMIT_FILE)

# Step 1: StartSubmissionUpload
r1 = requests.post(
    'https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload',
    headers=_headers,
    json={'competitionName': COMPETITION, 'fileName': 'submission_tabnet.csv',
          'contentLength': _file_size})
print(f"Step 1: {r1.status_code}")
_resp = r1.json()
_blob_token = _resp.get('token', _resp.get('blobToken', ''))
_upload_url = _resp.get('createUrl', _resp.get('uploadUrl', ''))

# Step 2: Upload to GCS
with open(SUBMIT_FILE, 'rb') as f:
    r2 = requests.put(_upload_url, headers={'Content-Type': 'text/csv'}, data=f)
print(f"Step 2: {r2.status_code}")

# Step 3: CreateSubmission
r3 = requests.post(
    'https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission',
    headers=_headers,
    json={'competitionName': COMPETITION, 'blobToken': _blob_token,
          'submissionDescription': SUBMIT_DESC})
print(f"Step 3: {r3.status_code}")
print(f"Response: {r3.json()}")
print(f"\nSubmitted: {SUBMIT_DESC}")